In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_all_slides.pkl"

# Path to zarr
local_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")

In [ ]:
all_filenames = df_HE["filename"].tolist()
unique_wsi = set(all_filenames)

print("Number of unique wsi filenames: ",  len(unique_wsi))
print("Number of wsi filenames: ", len(all_filenames))

In [ ]:
with_features = False
if with_features:
    feature_result = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary.csv"
    df_feature_result = pd.read_csv(feature_result)
    df_feature = df_feature_result[df_feature_result['status'] == "feature extraction complete"]["wsi_path"].astype(str)
    with_features = set(df_feature)
    print("WSIs with features detected: ", len(with_features))
    all_filenames = list(with_features)

In [ ]:
from helper_functions import with_tissue_artifact

tissue_error = False
if tissue_error: 
    df_sub = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="error", version="default")
    all_filenames = df_sub["filename"].tolist()
    print("Number of slides with error during default tissue detection: ", len(all_filenames))

In [ ]:
from tissue_artifact_segmentation import SegmentMany

segmenter = SegmentMany(all_filenames, cache_file, local_dir, "artifact", version="threshold")